In [ ]:
import json
from typing import Any, Callable

from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelRequest,
    ModelResponse,
    wrap_model_call,
)
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from typing_extensions import NotRequired

load_dotenv()

<div style="font: bold 40px sans-serif;">实战案例-邮件助手 Mail Friend</div>

<br/>

下面创建一个完整的Agent示例，综合运用常见中间件。

我们计划开发一个邮件助手的Agent，名为MailFriend，作用是帮助用户查看邮箱，回复邮件。不过进行邮件操作之前需要验证用户的身份，验证通过后才可以进行邮件操作。


# 1.state
首先，我们要定义state，用来记录用户鉴权状态


In [ ]:
class AuthenticatedState(AgentState):
    """记录当前 thread 是否已完成邮箱鉴权。"""

    authenticated: NotRequired[bool]

# 2.tools
接着，我们定义3个工具：
- authenticate : 用来模拟用户邮箱鉴权
- check_inbox : 模拟收取邮件
- send_email : 模拟发送邮件


In [4]:
# 定义工具，用于用户邮箱鉴权
@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""

    # 定义变量，记录校验结果
    authenticated = False
    message = "Authentication failed"

    # 校验邮箱和密码
    if email == "huge@itcast.cn" and password == "123":
        authenticated = True
        message = "Successfully authenticated"

    # 返回校验结果
    return Command(
            update={
                "authenticated": authenticated,
                "messages": [
                    ToolMessage(message, tool_call_id=runtime.tool_call_id)
                ],
            }
        )

In [5]:
# 定义工具，收取和发送邮件
@tool
def check_inbox() -> str:
    """Read an email from the given address."""
    # 模拟收件箱邮件
    return [
        {
            "subject": "周末见个面？",
            "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
            "from": "jane@itcast.cn",
            "status": "unread"
        },
        {
            "subject": "周五会议",
            "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
            "from": "lixiaolong@itcast.cn",
            "status": "checked"
        }
    ]


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"

# 3.动态工具中间件
为了减少调用模型时传递的参数，我们通过自定义中间件实现动态工具：
- 在用户邮箱授权之前，只能看到authenticate工具
- 在用户邮箱授权之后，只能看到send_email和check_inbox工具


In [6]:
@wrap_model_call
def dynamic_tool_call(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Allow read inbox and send email tools only if user provides correct email and password"""

    # 读取授权状态
    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    # 重写工具列表
    request = request.override(tools=tools)
    return handler(request)

# 4.动态提示词中间件
由于业务分为两个阶段，所以每个阶段的提示词也不一样：
- 鉴权阶段：告知AI的核心工作是鉴权
- 工作阶段：告知AI的核心工作是处理邮件


In [7]:
from langchain.agents.middleware import dynamic_prompt

# 授权前，要求鉴定用户权限的系统提示词
unauthenticated_prompt = """You are a helpful email assistant.
    For system security protocols, you must authenticate user before any other interaction.
    """

# 授权后，邮件处理的系统提示词
authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."


# 在中间件中通过判断state的authenticated值来动态切换提示词
@dynamic_prompt
def dynamic_prompt_func(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")
    final_prompt = authenticated_prompt if authenticated else unauthenticated_prompt
    return final_prompt

# 5.Agent
接下来，我们创建Agent

In [8]:
# 初始化checkpointer
checkpointer = InMemorySaver()

agent = create_agent(
    "deepseek-chat",
    tools=[authenticate, check_inbox, send_email],
    state_schema=AuthenticatedState,
    checkpointer=checkpointer,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt_func,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            }
        )
    ],
)

# 6.HITl的stream模式

接下来我们测试下Agent效果，这里我们选择采用stream流式调用。在HITL存在时，stream采用的模式比较特殊，需要同时包含两种模式：
- messages : 以token方式返回AI生成的内容
- updates : 返回Agent调用时的每个步骤的完整信息，比如当出现interrupt时，完整展示interrupt信息

同时，当我们使用的LangGraph版本大于1.1时，我们还可以设定stream的version为v2，这样返回的结果格式会以dict返回，而不是tuple，会更好处理。


In [ ]:
config = {"configurable": {"thread_id": "1"}}

response = agent.stream(
    {"messages": [HumanMessage("帮我查看下邮件。")]},
    config=config,
    stream_mode=["messages", "updates"],
    version="v2",
)


def print_chunk(chunk: dict[str, Any]) -> None:
    """打印 messages 流和 updates 流。"""
    chunk_type = chunk["type"]
    data = chunk["data"]

    if chunk_type == "messages":
        token, metadata = data
        if isinstance(token, AIMessage) and token.content:
            print(token.content, end="", flush=True)
    elif chunk_type == "updates":
        print("")
        print("================= updates =====================")
        print(data)


for chunk in response:
    print_chunk(chunk)

# 7.测试
## 7.1.邮箱鉴权

In [ ]:
response = agent.stream(
    {"messages": [HumanMessage("huge@itcast.cn/123")]},
    config=config,
    stream_mode=["messages", "updates"],
    version="v2",
)

for chunk in response:
    print_chunk(chunk)

## 7.2.触发Interrupt

In [ ]:
response = agent.stream(
    {
        "messages": [
            HumanMessage(
                "帮我回复Jane，很高兴她能来，问问她具体见面时间"
            )
        ]
    },
    config=config,
    stream_mode=["messages", "updates"],
    version="v2",
)

for chunk in response:
    print_chunk(chunk)

## 7.3.用户处理Interrupt

Agent 暂停后，用户可以选择：

| Decision | 效果 |
|---------|------|
| `approve` | 按原参数执行工具 |
| `reject` | 取消工具调用，并把拒绝原因返回给模型 |
| `edit` | 修改工具参数后再执行 |

下面先用 `reject` 演示。其余类型只需替换 `decisions` 中的对象。

### reject

In [12]:
# 用户处理interrupt，比如reject
response = agent.stream(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # 告知AI拒绝的原因
                    "message": "语气太书面了，这不是工作邮件，Jane是我的好朋友，回复自然一点。"
                }
            ]
        }
    ),
    config = config,
    stream_mode= ["messages", "updates"],
    version="v2"
)

for chunk in response:
    print_chunk(chunk)


=================updates=====================
{'HumanInTheLoopMiddleware.after_model': {'messages': [AIMessage(content='好的，我来回复Jane的邮件。', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'model_provider': 'deepseek'}, id='lc_run--019e0c1c-ea90-72d2-8d23-b0a843d6c6c1', tool_calls=[{'name': 'send_email', 'args': {'to': 'jane@itcast.cn', 'subject': 'Re: 周末见个面？', 'body': '嗨 简，\n\n很高兴你要来城里！我们当然可以一起喝杯咖啡。\n\n请问你具体什么时间方便见面呢？\n\n期待见面！\n\n祝好，\n虎哥'}, 'id': 'call_00_2WAI8S2tKU5FHdJW7PNL3802', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 876, 'output_tokens': 128, 'total_tokens': 1004, 'input_token_details': {'cache_read': 768}, 'output_token_details': {}}), ToolMessage(content='语气太书面了，这不是工作邮件，Jane是我的好朋友，回复自然一点。', name='send_email', id='b32acef4-fa75-45e5-892e-68ae067e16e1', tool_call_id='call_00_2WAI8S2tKU5FHdJW7PNL3802', status='error')]}

In [ ]:
def resume_with_decision(decision: dict[str, Any], config: dict) -> None:
    """使用指定决策恢复当前 thread 的 interrupt。"""
    response = agent.stream(
        Command(resume={"decisions": [decision]}),
        config=config,
        stream_mode=["messages", "updates"],
        version="v2",
    )
    for chunk in response:
        print_chunk(chunk)


# 批准原始工具调用：
# resume_with_decision({"type": "approve"}, config)

# 修改参数后再执行：
# resume_with_decision(
#     {
#         "type": "edit",
#         "edited_action": {
#             "name": "send_email",
#             "args": {
#                 "to": "jane@itcast.cn",
#                 "subject": "Re: 周末见个面？",
#                 "body": "很高兴见到你，具体时间你方便定吗？",
#             },
#         },
#     },
#     config,
# )

# 8.会话历史

`checkpointer.get(config)` 更偏向读取 checkpoint 原始数据。需要同时查看当前 State
和待处理 interrupt 时，使用 `agent.get_state(config)` 更直接。

In [13]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-05-09T09:41:36.374100+00:00',
 'id': '1f14b8b4-57fd-6d94-8011-d3cf9dd664b9',
 'channel_versions': {'__start__': '00000000000000000000000000000016.0.8655555881670611',
  'messages': '00000000000000000000000000000019.0.7544398364008247',
  'branch:to:model': '00000000000000000000000000000019.0.7544398364008247',
  'branch:to:HumanInTheLoopMiddleware.after_model': '00000000000000000000000000000019.0.7544398364008247',
  '__pregel_tasks': '00000000000000000000000000000012.0.2455949578797224',
  'authenticated': '00000000000000000000000000000009.0.4573035367639653'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000015.0.15918202817556004'},
  'model': {'branch:to:model': '00000000000000000000000000000018.0.38132983800910747'},
  'HumanInTheLoopMiddleware.after_model': {'branch:to:HumanInTheLoopMiddleware.after_model': '00000000000000000000000000000017.0.31280029323537684'},
  'tools': {},
  '__interrupt__': {'messages': '0

如果要获取到历史消息、interrupt等信息，必须通过AgentState：


In [41]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='帮我查看下邮件。', additional_kwargs={}, response_metadata={}, id='9d0e51c5-f322-4c9e-88fb-5a5b43115374'), AIMessage(content='好的，我先帮您进行身份验证。请问您的邮箱地址和密码是什么？', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'model_provider': 'deepseek'}, id='lc_run--019e0186-8a2c-74a0-9704-f14ab790d49b', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 316, 'output_tokens': 18, 'total_tokens': 334, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}), HumanMessage(content='huge@itcast.cn/123', additional_kwargs={}, response_metadata={}, id='30ce13f3-fb3b-4595-a459-c1c8a858ac57'), AIMessage(content='好的，我来帮您进行身份验证。', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'model_

定义处理消息的方法：

In [20]:
def get_messages(thread_id: str) -> dict:
    """获取会话历史，如果存在中断则返回中断信息"""

    config = {"configurable": {"thread_id": thread_id}}
    state = agent.get_state(config)
    if state is None or not state.values:
        return {"messages": []}

    messages = state.values.get("messages", [])

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue
        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    response = {"messages": result}

    # 检查是否存在中断
    interrupts = None
    if hasattr(state, 'interrupts') and state.interrupts:
        interrupts = state.interrupts
    elif hasattr(state, 'tasks') and state.tasks:
        for task in state.tasks:
            if hasattr(task, 'interrupts') and task.interrupts:
                interrupts = task.interrupts
                break

    if interrupts:
        response["has_interrupt"] = True
        response["interrupt"] = {
            "reason": "需要人工确认",
            "details": _serialize(interrupts)
        }

    return response

def _serialize(obj):
    """递归转换对象为可 JSON 序列化的格式"""
    if hasattr(obj, 'value'):
        return _serialize(obj.value)
    elif hasattr(obj, 'model_dump'):
        return obj.model_dump()
    elif isinstance(obj, (list, tuple)):
        return [_serialize(item) for item in obj]
    elif isinstance(obj, dict):
        return {k: _serialize(v) for k, v in obj.items()}
    return obj


In [ ]:
get_messages("1")